# conv-windowing-2d — faded example 3: Complete the einsum that contracts windows with the kernel

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-2d`. The last cell reports your progress on the `CNN: 2-D conv windowing` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 2-D conv windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-windowing-2d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-windowing-2d"
DD_SUBTOPIC = "CNN: 2-D conv windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Once the `(B, IC, OH, OW, KH, KW)` window view exists, a 2-D conv is one `einsum` against a kernel `(OC, IC, KH, KW)`: sum the elementwise product over the input-channel and kernel-spatial axes, keeping `(B, OC, OH, OW)`.

## Faded exercise 3

### Faded — finish the contraction

The window view is already built for you. Implement `conv2d_contract(windows, w)` by writing the **single `einsum` call** that contracts the windows against the kernel to produce `(B, OC, OH, OW)`, matching `F.conv2d`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from einops import einsum

def conv2d_contract(windows: Tensor, w: Tensor) -> Tensor:
    # windows: (B, IC, OH, OW, KH, KW), w: (OC, IC, KH, KW)
    return einsum(windows, w, 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')


import torch.nn.functional as F

def _test():
    t.manual_seed(3)
    x = t.randn(2, 3, 8, 9)
    KH, KW = 3, 2
    w = t.randn(5, 3, KH, KW)
    B, IC, H, W = x.shape
    OH, OW = H - KH + 1, W - KW + 1
    s_b, s_ic, s_h, s_w = x.stride()
    windows = x.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h, s_w, s_h, s_w),
    )
    out = conv2d_contract(windows, w)
    ref = F.conv2d(x, w)
    assert out.shape == ref.shape, (out.shape, ref.shape)
    assert t.allclose(out, ref, atol=1e-4), (out - ref).abs().max().item()


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from einops import einsum

def conv2d_contract(windows: Tensor, w: Tensor) -> Tensor:
    # windows: (B, IC, OH, OW, KH, KW), w: (OC, IC, KH, KW)
    return einsum(windows, w, 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')
```
</details>